In [1]:
import torch
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
from efficient_kan import KAN

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

cuda
NVIDIA GeForce GTX 1650 Ti


In [3]:
data = load_breast_cancer()

X = data.data
y = data.target

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32,
    device=device
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32,
    device=device
)

y_train = torch.tensor(
    y_train.reshape(-1,1),
    dtype=torch.float32,
    device=device
)

y_test = torch.tensor(
    y_test.reshape(-1,1),
    dtype=torch.float32,
    device=device
)

In [7]:
dataset = {
    'train_input': X_train,
    'train_label': y_train,
    'test_input': X_test,
    'test_label': y_test
}

In [8]:
model = KAN(
    layers_hidden=[30, 16, 8, 1]
).to(device)

In [9]:
import torch
from torch import nn

model = KAN(layers_hidden=[30,16,8,1]).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 100

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(X_train)

    loss = criterion(outputs, y_train)

    loss.backward()

    optimizer.step()

    print(epoch, loss.item())

0 0.6946787238121033
1 0.6915013194084167
2 0.688349723815918
3 0.6852141618728638
4 0.6820837259292603
5 0.6789491176605225
6 0.6758020520210266
7 0.6726346611976624
8 0.6694395542144775
9 0.6662097573280334
10 0.6629383563995361
11 0.6596183776855469
12 0.6562431454658508
13 0.6528060436248779
14 0.649301290512085
15 0.645723283290863
16 0.6420665383338928
17 0.6383258104324341
18 0.6344964504241943
19 0.630573570728302
20 0.6265526413917542
21 0.6224297285079956
22 0.6182003617286682
23 0.6138604879379272
24 0.6094061732292175
25 0.6048332452774048
26 0.6001387238502502
27 0.5953196287155151
28 0.5903740525245667
29 0.5853001475334167
30 0.5800970196723938
31 0.5747637748718262
32 0.5693005919456482
33 0.5637074708938599
34 0.5579853653907776
35 0.552136242389679
36 0.5461623668670654
37 0.540066659450531
38 0.5338525772094727
39 0.5275241732597351
40 0.5210860371589661
41 0.5145432353019714
42 0.5079013705253601
43 0.5011664032936096
44 0.49434447288513184
45 0.4874420166015625
46 

In [10]:
model.eval()

with torch.no_grad():

    outputs = model(X_test)

    pred = torch.sigmoid(outputs)

    pred = (pred > 0.5).float()

In [11]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test.cpu(),
    pred.cpu()
)

print(accuracy)

0.9649122807017544


In [12]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Convert tensors to CPU NumPy arrays
y_true = y_test.cpu().numpy().ravel().astype(int)
y_pred = pred.cpu().numpy().ravel().astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[40  2]
 [ 2 70]]


In [13]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=["Malignant", "Benign"]
))

              precision    recall  f1-score   support

   Malignant       0.95      0.95      0.95        42
      Benign       0.97      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.96      0.96       114
weighted avg       0.96      0.96      0.96       114

